# 강의 06 · 실습 2 — 패턴 2 라우팅 · (3) 변형

## 1. 문제상황

- 구립 도서관 안내 데스크에는 대출 연장, 스터디룸 예약, 독서 강좌 신청 같은 문의가 한 창구로 들어옵니다.
- 안내 담당자는 문의를 읽고 대출 담당·시설 담당·행사 담당 중 누가 답해야 하는지 고른 뒤 넘깁니다.
- 셋 중 어디에도 속하지 않는 문의는 종합 안내 담당이 받아 안내하거나 담당 기관을 알려 줍니다.
- 어느 담당이 답했든 문의와 답은 민원 대장에 남겨야 하는데, 담당마다 대장에 적는 일을 잊는 경우가 생깁니다.

## 2. 문제와 목표

- **문제**: 문의가 어느 창구의 일인지 사람이 고르고, 어느 창구에도 속하지 않는 문의는 처리가 멈추며, 대장 기록은 담당자마다 빠뜨립니다. 세 가지 일이 문의 수만큼 반복됩니다.
- **목표**: 문의를 입력하면 분류 노드가 대출·시설·행사·기타 중 한 창구를 고정된 형식으로 고르고, 조건부 엣지가 전담 노드로 보내고, 어느 전담 노드를 거치든 기록 노드에서 합류해 대장에 남기는 처리 흐름을 만듭니다.
    - 분류 노드: classify — 문의를 대출·시설·행사 중 하나로 고르되, 셋 중 어디에도 아니면 기타(구조화 출력 `Desk`).
    - 전담 노드 넷: loan(대출), facility(시설), event(행사), general(기타 — 예비 창구).
    - 기록 노드: record — 모델을 부르지 않고 배정 창구와 답 앞부분을 출력한 뒤 `logged`에 `True`를 씁니다.
- **목표 달성 여부의 판정 기준**: 대출 문의, 시설 문의, 어느 창구에도 속하지 않는 문의를 차례로 입력했을 때, 세 입력이 서로 다른 전담 노드를 거치고 세 입력 모두 마지막에 기록 노드를 거치는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex02_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 분류 규격을 정의합니다.**
    - 문의(`question`), 배정 창구(`desk`), 담당 창구의 답(`answer`), 대장 기록 여부(`logged`) 키 네 개를 가지는 상태를 선언합니다.
    - 분류 결과의 규격 `Desk`는 `name` 필드 하나를 가지며, 값은 대출·시설·행사·기타 넷 중 하나로만 제한합니다.
2. **분류 노드를 만듭니다.**
    - classify 노드는 `Desk` 규격을 건 모델을 불러 문의를 대출·시설·행사 중 한 창구로 배정하되, 셋 중 어디에도 속하지 않으면 기타로 배정하고, 그 값을 상태의 `desk` 키에 씁니다.
3. **창구 노드 네 개를 만듭니다.**
    - loan 노드는 대출 창구로서 대출 기간과 연장 방법을, facility 노드는 시설 창구로서 예약 방법과 이용 시간을, event 노드는 행사 창구로서 신청 방법과 마감일을, general 노드는 종합 안내 창구로서 도서관에서 답할 수 있는 범위와 담당 기관을 각각 두 문장으로 답해 상태의 `answer` 키에 씁니다.
4. **기록 노드를 만듭니다.**
    - record 노드는 모델을 부르지 않고, 배정 창구와 답의 앞부분을 화면에 출력한 뒤 상태의 `logged` 키에 `True`를 씁니다.
5. **그래프에 노드를 등록합니다.**
    - 여섯 노드를 이름과 함께 그래프에 등록합니다.
6. **엣지를 연결합니다.**
    - START에서 classify로 가는 고정 엣지를 추가합니다.
    - classify 뒤에는 `desk` 값을 보고 loan·facility·event·general 중 하나를 고르는 조건부 엣지를 추가합니다.
    - 네 창구 노드 뒤에는 각각 record를, record 뒤에는 END를 고정 엣지로 연결합니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 대출 문의, 시설 문의, 어느 창구에도 속하지 않는 문의를 차례대로 넣고, 노드가 하나 끝날 때마다 어느 노드가 상태의 어느 키를 채웠는지 화면에 출력한 뒤, 배정 창구와 기록 여부를 출력합니다.
    - 값(`QUESTIONS`, 문의 목록)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키와 분류 결과의 규격을 선언합니다 | `class LibraryState(TypedDict)`, `class Desk(BaseModel)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `llm.with_structured_output(Desk)`, `def classify(state) -> dict` | 2, 3, 4 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(LibraryState)`, `add_node` | 5 |
| ④ 엣지 연결 | 분류 결과에 따라 나뉘고 기록 노드에서 모이는 분기를 지정합니다 | `add_edge`, `add_conditional_edges` | 6 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field   # Field — 규격 필드에 설명(description=…)을 달 때 씁니다

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료: 문의 목록 QUESTIONS — 값을 그대로 씁니다
QUESTIONS = [
    "빌린 책 반납일이 내일인데 한 주 더 볼 수 있을까요?",
    "이번 주 토요일 오후에 스터디룸을 쓰고 싶은데 예약은 어떻게 하나요?",
    "도서관 옆 공영 주차장 요금은 얼마인가요?",
]

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. 상태와 함께 분류 결과의 규격 `Desk`도 여기서 선언합니다. `Literal`에 적힌 값 밖으로는 분류 결과가 나오지 않으므로, 어느 창구로 보낼지의 판단 기준이 스키마 한 곳에 모입니다. 넷째 값 「기타」는 셋 중 어디에도 속하지 않는 문의를 받는 예비 창구입니다.

In [ ]:
# 여기에 단계 ①(상태 정의와 분류 규격 Desk 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다. 돌려준 딕셔너리가 상태의 해당 키를 덮습니다.
- 분류 노드는 `llm.with_structured_output(Desk)`로 감싼 모델을 부릅니다. 돌아오는 값은 문자열이 아니라 `Desk` 객체이며, 분류 노드는 그 값을 상태의 `desk` 키에 넣기만 합니다.
- 창구 노드들은 같은 문의를 받아 같은 `answer` 키에 씁니다. 다른 것은 시스템 프롬프트뿐입니다. 창구끼리는 서로를 부르지 않습니다. 어느 창구가 실행될지는 함수 밖의 조건부 엣지가 정합니다.
- record 노드는 모델을 부르지 않습니다. 어느 창구를 거쳤든 같은 기록을 남깁니다.

In [ ]:
# 여기에 단계 ②(분류 노드, 창구 노드 네 개, 기록 노드 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 5)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 여기서 붙인 이름은 뒤의 엣지 연결에서 그대로 쓰입니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 6)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드가 나뉘는 분기를 추가합니다. 판단 함수 `route_decision`은 상태의 `desk` 키만 보고 갈 곳의 노드 이름을 돌려주며, 상태를 바꾸지 않습니다. 세 번째 인자는 갈 수 있는 노드 이름의 목록입니다. 네 창구 노드 뒤에는 END가 아니라 record를 연결합니다. 나뉜 분기가 다시 모이는 곳입니다.

In [ ]:
# 여기에 단계 ④(판단 함수와 엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `stream`은 노드가 하나 끝날 때마다 그 노드가 바꾼 키를 내보냅니다. 아래에서는 대출 문의, 시설 문의, 어느 창구에도 속하지 않는 문의를 차례대로 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 1번 문의(반납 연장)에서는 `classify`, `loan`, `record` 세 노드가 차례대로 출력되고, 2번 문의(스터디룸)에서는 `classify`, `facility`, `record`가 출력됩니다. 가운데 노드만 다릅니다.
2. 3번 문의(주차장 요금)는 대출·시설·행사 어디에도 속하지 않으므로 `desk` 값이 기타이고, `general` 노드가 출력됩니다. 예비 창구가 문의를 받습니다.
3. 세 문의 모두 마지막 줄의 `logged` 값이 `True`입니다. 어느 창구를 거쳤든 record 노드는 전부 거칩니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. `logged`가 `None`이면 창구 노드 뒤의 엣지가 record로 가는지 단계 ④를 다시 봅니다.